## Metrics Evaluation Lab

Throughout your early career as a Data Scientist you've spent most your time cleaning data, but now you are starting to build models and have come to realize the most important part about understanding any machine learning model (or any model, really) is understanding its weakness and vulnerabilities.

In doing so you've decided to practice on a dataset about mushrooms, because after all if you don't know how to evaluate a model thoroughly you'll be in real **truffle** (ha...ha) and introduce a approach that is new to you. 

Below I've build an initial **Decision Tree** model on the mushroom dataset to get you started. Think of the Decision Tree as your field guide – splitting mushrooms into categories one feature at a time. 

### Part 1: Load and Clean

Using the [mushroom dataset](https://archive.ics.uci.edu/static/public/848/secondary+mushroom+dataset.zip) and the documentation below answer the provided question. 

- [Mushroom Documentation](https://archive.ics.uci.edu/dataset/848/secondary+mushroom+dataset)

*How well can we predict whether a mushroom is poisonous or edible based on its physical characteristics?* 

### Part 2: Build the model

This will most be provided for you, but some details you'll need to code yourself. 

### Part 3: Evaluate and assess

Consider where classification errors are occurring, is there a pattern? If so discuss this pattern and why you think this is the case. Use the confusion matrix to determine the pattern. 

### Keys to Success

- Using the evaluation metrics correctly: we are focusing on classification not regression
- Evaluation is not about the metrics per se, but what they mean; speaking through your question in light of the evaluation metrics is the primary objective of this lab. Think of yourself as a "model detective" that works to leave no stone unturned!
- Remember, be patient and double check your code or you might find yourself in real **shiitake** :)

---
## Initial Decision Tree Model – Starter Code

The code below walks you through an initial Decision Tree classifier on the mushroom dataset. Use it as a *spore*-ing board (sorry, not sorry) to complete the lab.

### Step 1 – Imports

In [1]:
# Imports 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn import metrics

### Step 2 – Load the Mushroom Dataset

We load the mushroom dataset directly. The target column is `type` (poisonous **p** vs edible **e**). 

For the full lab you may want to use the secondary mushroom dataset from UCI:
- [Download](https://archive.ics.uci.edu/static/public/848/secondary+mushroom+dataset.zip)
- [Documentation](https://archive.ics.uci.edu/dataset/848/secondary+mushroom+dataset)

In [39]:
# Load the mushroom dataset
mushroom_url = (
    "https://raw.githubusercontent.com/stedy/"
    "Machine-Learning-with-R-datasets/master/mushrooms.csv"
)

mushroom_data = pd.read_csv(mushroom_url)

print(f'Dataset shape: {mushroom_data.shape}')
mushroom_data.head()

Dataset shape: (8124, 23)


,type,cap_shape,cap_surface,cap_color,bruises,odor,gill_attachment,gill_spacing,gill_size,gill_color,...,stalk_surface_below_ring,stalk_color_above_ring,stalk_color_below_ring,veil_type,veil_color,ring_number,ring_type,spore_print_color,population,habitat
0,p,x,s,n,t,p,f,c,n,k,...,s,w,w,p,w,o,p,k,s,u
1,e,x,s,y,t,a,f,c,b,k,...,s,w,w,p,w,o,p,n,n,g
2,e,b,s,w,t,l,f,c,b,n,...,s,w,w,p,w,o,p,n,n,m
3,p,x,y,w,t,p,f,c,n,n,...,s,w,w,p,w,o,p,k,s,u
4,e,x,s,g,f,n,f,w,b,k,...,s,w,w,p,w,o,e,n,a,g


### Step 3 – Explore the Data

In [40]:
# Use the various exploration methods we covered in class to understand the dataset.
print(mushroom_data.dtypes)
print(mushroom_data.describe())

type                        str
cap_shape                   str
cap_surface                 str
cap_color                   str
bruises                     str
odor                        str
gill_attachment             str
gill_spacing                str
gill_size                   str
gill_color                  str
stalk_shape                 str
stalk_root                  str
stalk_surface_above_ring    str
stalk_surface_below_ring    str
stalk_color_above_ring      str
stalk_color_below_ring      str
veil_type                   str
veil_color                  str
ring_number                 str
ring_type                   str
spore_print_color           str
population                  str
habitat                     str
dtype: object
        type cap_shape cap_surface cap_color bruises  odor gill_attachment  \
count   8124      8124        8124      8124    8124  8124            8124   
unique     2         6           4        10       2     9               2   
top        e    

In [41]:
print(mushroom_data.info())
mushroom_data.head()

<class 'pandas.DataFrame'>
RangeIndex: 8124 entries, 0 to 8123
Data columns (total 23 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   type                      8124 non-null   str  
 1   cap_shape                 8124 non-null   str  
 2   cap_surface               8124 non-null   str  
 3   cap_color                 8124 non-null   str  
 4   bruises                   8124 non-null   str  
 5   odor                      8124 non-null   str  
 6   gill_attachment           8124 non-null   str  
 7   gill_spacing              8124 non-null   str  
 8   gill_size                 8124 non-null   str  
 9   gill_color                8124 non-null   str  
 10  stalk_shape               8124 non-null   str  
 11  stalk_root                8124 non-null   str  
 12  stalk_surface_above_ring  8124 non-null   str  
 13  stalk_surface_below_ring  8124 non-null   str  
 14  stalk_color_above_ring    8124 non-null   str  
 15

,type,cap_shape,cap_surface,cap_color,bruises,odor,gill_attachment,gill_spacing,gill_size,gill_color,...,stalk_surface_below_ring,stalk_color_above_ring,stalk_color_below_ring,veil_type,veil_color,ring_number,ring_type,spore_print_color,population,habitat
0,p,x,s,n,t,p,f,c,n,k,...,s,w,w,p,w,o,p,k,s,u
1,e,x,s,y,t,a,f,c,b,k,...,s,w,w,p,w,o,p,n,n,g
2,e,b,s,w,t,l,f,c,b,n,...,s,w,w,p,w,o,p,n,n,m
3,p,x,y,w,t,p,f,c,n,n,...,s,w,w,p,w,o,p,k,s,u
4,e,x,s,g,f,n,f,w,b,k,...,s,w,w,p,w,o,e,n,a,g


In [42]:
# Check for missing values
print(mushroom_data.isna().sum())

type                        0
cap_shape                   0
cap_surface                 0
cap_color                   0
bruises                     0
odor                        0
gill_attachment             0
gill_spacing                0
gill_size                   0
gill_color                  0
stalk_shape                 0
stalk_root                  0
stalk_surface_above_ring    0
stalk_surface_below_ring    0
stalk_color_above_ring      0
stalk_color_below_ring      0
veil_type                   0
veil_color                  0
ring_number                 0
ring_type                   0
spore_print_color           0
population                  0
habitat                     0
dtype: int64


No missing values :D

### Step 4 – Clean & Prepare the Data



In [43]:
# Separate features (X) and target (y)
# Our target is 'type' – predicting poisonous (p) vs edible (e)
y = mushroom_data['type']
X = mushroom_data.drop('type', axis=1).copy()

In [44]:
# Calculate prevalence – how common is each class?
# This tells us what accuracy we'd get if we just guessed the majority class every time.
# If our model can't beat the prevalence, it's about as useful as a chocolate teapot...
# or a poisonous mushroom at a dinner party 
prev = y.value_counts(normalize=True)
print(f'Prevalence: {prev}')

Prevalence: type
e    0.517971
p    0.482029
Name: proportion, dtype: float64


The pervalence of edible mushrooms is about 52% and poisonous is about 48%. This is quite balanced out, so a model that would just guess the majority class every time would only be correct about 52% of the time, which means a guessing model wouldn't perform very well. So our model needs to be meaningfully better than essentially just flipping a coin. Since it's balanced, it helps us in terms of weighting and stratifying though since they're so balanced, it's just really nice for our model building.

### Collapse factor levels


Check for the levels of the categorical features
This helps us understand the diversity of our features and how they might influence the model 
and if we need to collapse rare categories to avoid overfitting on tiny groups of mushrooms, don't use 
a for loop, `nunique()` will give us the unique values for each column at once, its a function of the dataframe, 
not a function of the column, so we can call it on the whole dataframe and it will return the 
unique values for each column in one go


Then use data wrangler or another method to identify features that need to be collapsed, 
for example if a feature has 10 levels but 9 of them are very rare, 
we might want to collapse those 9 levels into an "other" 
category to avoid overfitting on those rare categories, this is especially 
important for decision trees which can easily overfit on rare categories

In [45]:
# Collapse rare categories in the 'habitat' feature
# This is a common preprocessing step to avoid overfitting on rare categories,
# especially important for decision trees which can easily overfit on rare categories
# X['habitat'] = X['habitat'].replace(['u', 'd'], 'other') # in this example we are simply replacing the 'u' and 'd' 

# categories with 'other', but in a real analysis we would want to look at the distribution of the categories and 
# decide which ones to collapse based on their frequency and importance to the model, 
# this is just an example to show how to do it, 
# you would need to adjust it based on your specific dataset and analysis needs

# Other functions that might be helpful for this step include value_counts(), where() and isin() 
# sample code for how that might look, but again you would need to adjust it based on your specific dataset and analysis needs

# top_cats = df['col'].value_counts().nlargest(2).index # this would give us the top 2 most common categories in the 'col' feature,
# df['new_col'] = df['col'].where(df['col'].isin(top_cats), 'Other') # this would create a new column 'new_col' where the 
# values are the same as 'col' for the top 2 categories,and 'Other' for all other categories,
# this is a common way to collapse rare categories into an 'Other' category,
# but again you would need to adjust it based on your specific dataset and analysis needs.

In [46]:
X.nunique().sort_values()

veil_type                    1
bruises                      2
gill_spacing                 2
gill_size                    2
stalk_shape                  2
gill_attachment              2
ring_number                  3
cap_surface                  4
stalk_surface_above_ring     4
stalk_surface_below_ring     4
veil_color                   4
stalk_root                   5
ring_type                    5
population                   6
cap_shape                    6
habitat                      7
odor                         9
spore_print_color            9
stalk_color_above_ring       9
stalk_color_below_ring       9
cap_color                   10
gill_color                  12
dtype: int64

In [47]:
X = X.drop(columns = ['veil_type']) # this feature is just the same thing over and over (no information)

In [48]:
# collapse rare factor levels (only take 3 largest and label rest as other)
top_stalk_root = X['stalk_root'].value_counts().nlargest(3).index
X['stalk_root'] = X['stalk_root'].where(X['stalk_root'].isin(top_stalk_root), 'other')

top_spore = X['spore_print_color'].value_counts().nlargest(5).index
X['spore_print_color'] = X['spore_print_color'].where(X['spore_print_color'].isin(top_spore), 'other')

top_cap_color = X['cap_color'].value_counts().nlargest(5).index
X['cap_color'] = X['cap_color'].where(X['cap_color'].isin(top_cap_color), 'other')

top_gill_color = X['gill_color'].value_counts().nlargest(5).index
X['gill_color'] = X['gill_color'].where(X['gill_color'].isin(top_gill_color), 'other')

top_habitat = X['habitat'].value_counts().nlargest(5).index
X['habitat'] = X['habitat'].where(X['habitat'].isin(top_habitat), 'other')

top_odor = X['odor'].value_counts().nlargest(5).index
X['odor'] = X['odor'].where(X['odor'].isin(top_odor), 'other')

top_stalk_color_above_ring = X['stalk_color_above_ring'].value_counts().nlargest(5).index
X['stalk_color_above_ring'] = X['stalk_color_above_ring'].where(X['stalk_color_above_ring'].isin(top_stalk_color_above_ring), 'other')

top_stalk_color_below_ring = X['stalk_color_below_ring'].value_counts().nlargest(5).index
X['stalk_color_below_ring'] = X['stalk_color_below_ring'].where(X['stalk_color_below_ring'].isin(top_stalk_color_below_ring), 'other')



In [49]:
X.nunique().sort_values()

bruises                     2
gill_size                   2
gill_spacing                2
gill_attachment             2
stalk_shape                 2
ring_number                 3
stalk_root                  4
cap_surface                 4
veil_color                  4
stalk_surface_below_ring    4
stalk_surface_above_ring    4
ring_type                   5
odor                        6
gill_color                  6
cap_shape                   6
cap_color                   6
stalk_color_below_ring      6
stalk_color_above_ring      6
spore_print_color           6
population                  6
habitat                     6
dtype: int64

### Step 5 – Partition the Data

We split our fungi into training, tuning, and test sets.

In [51]:
# 70 / 15 / 15 split – train / tune / test
# Stratify keeps the same ratio of edible:poisonous in each set, which is important for evaluation.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=21, stratify=y)
X_tune, X_test, y_tune, y_test = train_test_split(X_test, y_test, test_size=0.5, random_state=21, stratify=y_test)
print(f'Train set shape: {X_train.shape}, Tune set shape: {X_tune.shape}, Test set shape: {X_test.shape}')
print(f'Train set: {y_train.value_counts()}')
print(f'Tune set: {y_tune.value_counts()}')
print(f'Test set: {y_test.value_counts()}')

Train set shape: (5686, 21), Tune set shape: (1219, 21), Test set shape: (1219, 21)
Train set: type
e    2945
p    2741
Name: count, dtype: int64
Tune set: type
e    632
p    587
Name: count, dtype: int64
Test set: type
e    631
p    588
Name: count, dtype: int64


In [ ]:
categorical_features = X.select_dtypes(include='object').columns.tolist() # creating a list of the categorical features, 
# this will be used later for preprocessing in the pipeline, in this example we are using a ordinal encoder 
# which can handle categorical features without needing to one-hot encode them,
# but we need to tell the pipeline which features are categorical so it knows to apply the ordinal encoder to 
# those features and not to the numeric features (if we had any).


# Create a LabelEncoder for the target variable
le_target = LabelEncoder()
le_target.fit(y)

# Build decision tree pipeline with OrdinalEncoder preprocessing
dt_pipeline = Pipeline([
    ('preprocessor', ColumnTransformer( #note the preprocessor and the classifier on the same level of the pipeline, 
        #this is important because we want to make sure that the preprocessing is applied to the 
        # training data during cross-validation and grid search, if we put the preprocessor inside the classifier 
        # it would only be applied to the test data and not the training data, which would lead to data leakage and overfitting
        transformers=[
            ('ordinal', OrdinalEncoder(), categorical_features) # applying the ordinal encoder to the categorical features, 
            #this will convert the categorical features into numeric 
            # values that can be used by the decision tree classifier
        ],
        remainder='passthrough'
    )),
    ('classifier', DecisionTreeClassifier(random_state=42, criterion='gini'))
])

# Evaluate with 5-fold cross-validation on training data
cv_scores = cross_val_score(
    dt_pipeline,
    X_train,
    y_train,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)


# Fit final model on full training data



Pipeline Training accuracy: 0.9773
Pipeline Tuning accuracy:   0.9869


---
## Evaluation Metrics

Time to find out how well our tree knows its fungi.

In [ ]:
# Generate predictions and predicted probabilities on the tuning set and pass into variables
# Think of probabilities as how confident our tree is, example below
dt_pred = dt_model.predict(X_tune)
dt_prob = dt_model.predict_proba(X_tune)

# Bundle everything into a tidy DataFrame
results = pd.DataFrame({
    'target': y_tune,
    'pred': dt_pred,
    'prob_edible': dt_prob[:, 0], # first column
    'prob_poisonous': dt_prob[:, 1] # second column
})


#### Confusion Matrix

The confusion matrix shows us where our model is getting confused, build a confusion matrix and see how the model is performing. Give a 2 sentence summary on what you see. 

#### True Positive Rate (Sensitivity / Recall) & False Positive Rate

In [ ]:
# TPR (True Positive Rate) = Recall = Sensitivity
# Of all the actually poisonous mushrooms, how many did we correctly flag?
# FPR (False Positive Rate) = how many safe mushrooms did we wrongly accuse?

#### Classification Report

In [ ]:
# Full classification report – the tasting menu of evaluation metrics 


#### ROC Curve & AUC

The ROC curve plots TPR vs FPR at every threshold. AUC (Area Under the Curve) summarises it in one number – 1.0 is perfect, 0.5 is a coin flip.  

### Write a summary of what you found based on the evaluation measures. Include where have you noticed some issues with the model and What metrics do you think we should pay most attention to given the target variable.

 - Do you think we should adjust the threshold from the default value or not? Why or why not?
 -  

---

You now have a working Decision Tree baseline.

**Attempt the Bonus** 5 points each:

Pick a metric we haven’t covered (e.g., Matthews Correlation Coefficient, Precision-Recall AUC, Cohen’s Kappa) and discuss it.

Choose a model we have not present in class and see if the evaluation is better or worse - be specific about the metric you are using for comparison and why the model seems to fit better or worse. 

